In [1]:
# =============================================================================
#  REGRESSION MODEL TRAINING
#  Project : Crop Yield Prediction Using Environmental and Nutrient Factors
#  Course  : Data Warehousing and Data Mining (DWDM)
#  Target  : Crop_Yield_kg_ha  (continuous — kg per hectare)
#
#  Models trained separately :
#    1. Linear Regression   (baseline — simple)
#    2. Ridge Regression    (baseline — regularised)
#    3. Decision Tree       (non-linear, interpretable)
#    4. Random Forest       (ensemble, best performer)
#
#  Each model is :
#    -> Trained on training set
#    -> Scored on training set AND test set
#    -> Cross-validated with 5-fold CV
#    -> Compared in a final summary table
# =============================================================================

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score, KFold

# LOAD DATA

train = pd.read_csv("crop_yield_train_preprocessed.csv")
test  = pd.read_csv("crop_yield_test_preprocessed.csv")

FEAT_COLS = [c for c in train.columns if c not in ["Crop_Yield_kg_ha", "Yield_Category"]]

X_train = train[FEAT_COLS].values
X_test  = test[FEAT_COLS].values
y_train = train["Crop_Yield_kg_ha"].values
y_test  = test["Crop_Yield_kg_ha"].values

In [3]:
def print_scores(name, y_tr, y_tr_pred, y_te, y_te_pred):
    train_r2   = r2_score(y_tr, y_tr_pred)
    test_r2   = r2_score(y_te, y_te_pred)
    test_rmse = np.sqrt(mean_squared_error(y_te, y_te_pred))
    test_mae  = mean_absolute_error(y_te, y_te_pred)
    test_mse = mean_squared_error(y_te,y_te_pred)
    gap     = abs(train_r2 - test_r2)
    fit_status = (
        "Good fit"     if gap < 0.03 else
        "Overfit"     if train_r2 > test_r2 else
        "Underfit"
    )
    return train_r2, test_r2, test_rmse, test_mae, test_mse

def print_cv(model, X, y, model_name, cv=5, sample=50000):
    kf  = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X[:sample], y[:sample],
                             cv=kf, scoring="r2", n_jobs=1)
    print(f"R² scores : {[round(s,4) for s in scores]}")
    print(f"  Mean R²  {scores.mean()}")
    print(f"  Std  R²        : {scores.std()}  "
          f"{'(stable)' if scores.std() < 0.02 else '(unstable)'}")
    return scores

In [4]:
summary = []

#  MODEL 1 — LINEAR REGRESSION

m1 = LinearRegression(n_jobs=-1)
print("  Training...")
m1.fit(X_train, y_train)

tr_pred1 = m1.predict(X_train)
te_pred1 = m1.predict(X_test)
print("Training complete\n")
tr_r2_1, te_r2_1, te_rmse_1, te_mae_1, te_mse_1 = print_scores(
    "Linear Regression", y_train, tr_pred1, y_test, te_pred1)

cv_scores_1 = print_cv(LinearRegression(n_jobs=-1), X_train, y_train, "Linear Regression")

summary.append(["Linear Regression",     tr_r2_1, te_r2_1, te_rmse_1, te_mae_1, te_mse_1, cv_scores_1.mean(), cv_scores_1.std()])

  Training...
Training complete

R² scores : [np.float64(0.9066), np.float64(0.897), np.float64(0.9001), np.float64(0.9038), np.float64(0.8975)]
  Mean R²  0.9010019565726118
  Std  R²        : 0.0036783169565012784  (stable)


In [5]:
#  MODEL 2 — RIDGE REGRESSION

m2 = Ridge(alpha=10.0)
print("  Training...")
m2.fit(X_train, y_train)

tr_pred2 = m2.predict(X_train)
te_pred2 = m2.predict(X_test)
print("Training complete\n")
tr_r2_2, te_r2_2, te_rmse_2, te_mae_2, te_mse_2 = print_scores(
    "Ridge Regression", y_train, tr_pred2, y_test, te_pred2)

cv_scores_2 = print_cv(Ridge(alpha=10.0), X_train, y_train, "Ridge Regression")

summary.append(["Ridge Regression",    tr_r2_2, te_r2_2, te_rmse_2, te_mae_2, te_mse_2, cv_scores_2.mean(), cv_scores_2.std()])

  Training...
Training complete

R² scores : [np.float64(0.9065), np.float64(0.897), np.float64(0.9002), np.float64(0.9037), np.float64(0.8975)]
  Mean R²  0.9009651622613155
  Std  R²        : 0.0036358786691928734  (stable)


In [6]:
#  MODEL 3 — DECISION TREE REGRESSOR


m3 = DecisionTreeRegressor(
    max_depth        = 12,
    min_samples_leaf = 10,
    min_samples_split= 20,
    random_state     = 42
)
print("  Training...")
m3.fit(X_train, y_train)

tr_pred3 = m3.predict(X_train)
te_pred3 = m3.predict(X_test)
print("  Training complete\n")
tr_r2_3, te_r2_3, te_rmse_3, te_mae_3, te_mse_3 = print_scores(
    "Decision Tree", y_train, tr_pred3, y_test, te_pred3)

cv_scores_3 = print_cv(
    DecisionTreeRegressor(max_depth=12, min_samples_leaf=10,
                          min_samples_split=20, random_state=42),
    X_train, y_train, "Decision Tree")

summary.append(["Decision Tree",     tr_r2_3, te_r2_3, te_rmse_3, te_mae_3, te_mse_3, cv_scores_3.mean(), cv_scores_3.std()])

  Training...
  Training complete

R² scores : [np.float64(0.96), np.float64(0.9624), np.float64(0.9631), np.float64(0.963), np.float64(0.9626)]
  Mean R²  0.9622231906635456
  Std  R²        : 0.0011132028869577386  (stable)


In [7]:
#  MODEL 4 — RANDOM FOREST REGRESSOR

m4 = RandomForestRegressor(
    n_estimators    = 100,
    max_depth       = 15,
    min_samples_leaf= 10,
    max_features    = "sqrt",
    n_jobs          = -1,
    random_state    = 42
)
print(" Training...\n")
m4.fit(X_train, y_train)

tr_pred4 = m4.predict(X_train)
te_pred4 = m4.predict(X_test)
print("Training complete\n")
tr_r2_4, te_r2_4, te_rmse_4, te_mae_4, te_mse_4 = print_scores(
    "Random Forest", y_train, tr_pred4, y_test, te_pred4)

cv_scores_4 = print_cv(
    RandomForestRegressor(n_estimators=30, max_depth=15,
                          min_samples_leaf=10, max_features="sqrt",
                          n_jobs=-1, random_state=42),
    X_train, y_train, "Random Forest")

summary.append(["Random Forest",     tr_r2_4, te_r2_4, te_rmse_4, te_mae_4, te_mse_4, cv_scores_4.mean(), cv_scores_4.std()])

Training complete

R² scores : [np.float64(0.9577), np.float64(0.9581), np.float64(0.9566), np.float64(0.96), np.float64(0.9565)]
  Mean R²  0.9577618176758262
  Std  R²        : 0.0012665998062922265  (stable)


In [13]:
#  FINAL COMPARISON TABLE

print(f"\n  {'Model':<22} {'Train R²':>9} {'Test R²':>9} {'RMSE':>9}"
      f" {'MAE':>9} {'MSE' :>9} {'CV Mean':>13} {'CV Std':>11} {'Status':>5}")

best_model_name = ""
best_r2 = -1
for row in summary:
    name, tr_r2, te_r2, rmse, mae, mse, cv_m, cv_s = row
    gap = abs(tr_r2 - te_r2)
    status = "Good fit" if gap < 0.03 else "Overfit" if tr_r2 > te_r2 else "Underfit"
    print(f"  {name:<22} {tr_r2:>9.4f} {te_r2:>9.4f} {rmse:>9.1f}"
          f" {mae:>9.1f} {mse:>9.4f} {cv_m:>9.4f} {cv_s:>8.4f} {status}")
    if te_r2 > best_r2:
        best_r2 = te_r2
        best_model_name = name

print(f"""
  Conclusion :
    Best model   -> {best_model_name}
    Test R²      -> {best_r2:.4f}""")


  Model                   Train R²   Test R²      RMSE       MAE       MSE       CV Mean      CV Std Status
  Linear Regression         0.9005    0.9025    3297.8    1486.8 10875426.2823    0.9010   0.0037 Good fit
  Ridge Regression          0.9005    0.9025    3297.7    1487.4 10874813.1904    0.9010   0.0036 Good fit
  Decision Tree             0.9789    0.9692    1855.0     714.8 3441152.5640    0.9622   0.0011 Good fit
  Random Forest             0.9719    0.9655    1963.1     788.0 3853786.1783    0.9578   0.0013 Good fit

  Conclusion :
    Best model   -> Decision Tree
    Test R²      -> 0.9692
